###  Project Title:PRCP-1024 – Texas Employee Salary Prediction
- The goal of this project is to analyze and predict the salary patterns of government employees working for the State of Texas. By using historical payroll data, the project aims to:

- Understand salary distributions and patterns across different job titles, departments, and demographic groups.

- Identify outliers (unusually high or low salaries).

- Analyze wage disparities — how pay differs between managers and non-managers, and between different departments.

- Develop a machine learning model to predict an employee’s salary based on their attributes (such as department, role, and experience).

### Task Description

### Data Preprocessing
1. Load and inspect the raw dataset.

2. Handle missing or null values.

3. Standardize column names and data types.

4. Convert date columns (e.g., EMPLOY DATE) into usable features (EMPLOY YEAR).

5. Drop unnecessary text fields (e.g., FIRST NAME, LAST NAME).

6. Identify and remove data leakage columns (e.g., MONTHLY, summed_annual_salary).

7. Encode categorical variables and prepare data for EDA.

### Exploratory Data Analysis (EDA)
1. Understand the distribution of salary data across agencies, roles, and demographics.

2. Visualize numeric and categorical features using histograms, boxplots, bar charts, and heatmaps.

3. Detect outliers in the ANNUAL column and understand their impact.

4. Explore salary trends based on employment year and work hours.

5. Check feature correlations and relationships with the target variable.

6. Document observations and patterns for feature selection and modeling.

### Modeling and Evaluation
1. Scale numeric features where needed (e.g., for KNN).

2. Split the data into training and testing sets.

3. Train multiple regression models

4. Evaluate models using R², MAE, and RMSE.

5. Tune hyperparameters for the best models (e.g., XGBoost and Random Forest).

6. Compare results and select the final model.

7. Visualize feature importance for interpretability.

###  Domain Analysis — Texas State Employee Salary

- This project belongs to the public sector payroll domain, focusing on the Texas state government.    
The state employs thousands of workers across various departments, with salaries determined by role, agency, and experience.  

- Accurate salary prediction helps in budget planning, transparency, and identifying wage disparities.   
Machine learning models can analyze payroll structures, detect outliers, and forecast future salaries.   
This supports data-driven decision-making for fair and efficient compensation management.  

### Task 1:-Prepare a complete data analysis report on the given data.

### Importing Libiraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

### Importing Files

In [ ]:
data = pd.read_csv("salary.csv")

### Data Preprocessing
Data preprocessing involves cleaning and preparing raw data to make it suitable for analysis. It includes handling missing values, encoding categorical features, removing outliers, and scaling numerical columns to ensure better model performance and accuracy.

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.nunique()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
print((data == 'unknown').sum())

### Handling  Missing Values
Missing values were identified and addressed to ensure data completeness. Depending on the column type, missing entries were either filled with appropriate statistical measures or removed to maintain dataset quality.

In [ ]:
#checking how many missing values each column has
missing_values = data.isnull().sum().sort_values(ascending=False)
print("missing values in each column:")
print(missing_values)

In [ ]:
#Checking the percentage of missing values
missing_percent = (data.isnull().mean() * 100).sort_values(ascending=False)
print("missing values in percentage:")
print(missing_percent.head(20))

In [ ]:
#drop columns with too many missing values
threshold = 0.5   #50%
data = data[data.columns[data.isnull().mean() < threshold]]
print("column after dropping high-null columns:")
print(data.columns.tolist())

In [ ]:
#drop rows where salary(annual)is missing(target column)
dat = data.dropna(subset=['ANNUAL'])
print("rows after dropping missing salary values:", data.shape)

In [ ]:
#fill missing categorical values with "unknown"
categorical_cols = ['agency name','class title', 'ethicity', 'gender','status']

for col in categorical_cols:
    if col in data.columns:
        data[col] = data[col].fillna('Unknown')

In [ ]:
# fill missing numeric values with median
numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

In [ ]:
#verify that missing values are handled
print("total missing values after cleaning:", data.isnull().sum().sum())

###  Convert Numeric Columns
Categorical or mixed-type columns containing numeric information were converted into proper numeric formats. This ensured accurate mathematical operations and model training without data type conflicts.

In [ ]:
#Define the columns you want to convert to numeric
numeric_cols = ['HRLY RATE', 'HRS PER WK', 'MONTHLY', 'ANNUAL']

In [ ]:
#convert each column to numeric type (invalid values will become NaN)
for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
#check the data types after conversion
print(data[numeric_cols].dtypes)

###  Fill Any NaNs After Conversion(if some rext turned into NaN)
After converting columns to numeric types, some non-numeric text entries turned into NaN. These missing values were handled using appropriate imputation techniques to maintain data consistency.

In [ ]:
for col in numeric_cols:
    if col in data.columns:
        data[col] = data[col].fillna(data[col].median())

In [ ]:
#double chech the data
print(data[numeric_cols].describe())

### EXPLPORATORY DATA ANALYSIS(EDA)
EDA was performed to understand the dataset’s structure, detect patterns, and identify outliers or missing values.   
It helped in gaining insights into salary distribution, feature relationships, and overall data quality.  

### 1.Basic Dataset Information

In [ ]:
# Shape and column info
print("Dataset Shape:", data.shape)
print("\n Columns:\n", data.columns.tolist())
# Gendral info
print("\n Dataset Info:")
data.info()
# Basic statistics of numeric columns
print("\n Numeric Summary:")
print(data.describe())

### 2.Check Unique Values In Categorical Columns

In [ ]:
categorical_cols = ['AGENCY NAME', 'CLASS TITLE', 'ETHNICITY', 'GENDER', 'STATUS']

for col in categorical_cols:
    if col in data.columns:
        print(f"\n {col.upper()} - unique values: {data[col].nunique()}")
        print(data[col].value_counts().head(10))   # top 10 categories

### 3.Salary Distrubution(Histogram)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10,5))
sns.histplot(data['ANNUAL'], bins=50, kde=True)
plt.title('Annual salary Distribution')
plt.xlabel('Annual salary')
plt.ylabel('Frequency')
plt.show()

The histogram shows how employee salaries are distributed across the dataset.     
It helps identify whether salaries are normally distributed or skewed toward higher or lower income ranges.

### 4. Boxplot For Outlier Visualization

In [ ]:
plt.figure(figsize=(10,4))
sns.boxplot(x=data['ANNUAL'])
plt.title('Boxplot of Annual Salary(Before outlier removal)')
plt.show()

The boxplot visually highlights the presence of outliers in salary data and other numerical features. It helps identify extreme values that may influence model performance and guides decisions on data cleaning or treatment.

### 5. Department-Wise Salary Overview(Top Agencies)

In [ ]:
plt.figure(figsize=(12,6))
top_agencies = data['AGENCY NAME'].value_counts().head(10).index
sns.boxplot(x='AGENCY NAME', y='ANNUAL', data=data[data['AGENCY NAME'].isin(top_agencies)])
plt.xticks(rotation=90)
plt.title('Salary Distribution by top 10 agencies')
plt.show()

This analysis shows the average salary distribution across top agencies or departments. It helps identify which agencies offer higher salaries and highlights pay disparities between different sectors.

### 6. Job Total Salary Overview

In [ ]:
data.columns

In [ ]:
plt.figure(figsize=(12,6))
top_titles = data['CLASS TITLE'].value_counts().head(10).index
sns.boxplot (x='CLASS TITLE', y='ANNUAL', data=data[data['CLASS TITLE'].isin(top_titles)])
plt.xticks(rotation=90)
plt.title('salary distribution by top 10 job titles')
plt.show()

This graph represents the total salary distribution across different job titles. It helps identify the highest and lowest-paying roles, offering insights into salary trends based on job positions.

### 7.Gender-Wise Salary Comparision

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='GENDER', y='ANNUAL',data=data)
plt.title('Gender-wise salary distribution')
plt.show()

This graph compares the average salaries between male and female employees. It highlights potential pay gaps and helps analyze gender-based salary distribution within the organization.

### 8.Correlation Heatmap(numeric columns)

In [ ]:
#select only numeric columns
numeric_data = data.select_dtypes(include=['float64', 'int64'])

In [ ]:
#compute correlation matrix
corr = numeric_data.corr()

In [ ]:
#plot heatmap
plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("correlation heatmap (numericcolumns only)")
plt.show()

The heatmap visualizes relationships between numerical features in the dataset. It helps identify which variables are strongly correlated with salary, aiding in better feature selection for modeling.

### 9.Salary Trends Over Time(if employ date is available)

In [ ]:
#convert employ date to datetime
data['EMPLOY DATE'] = pd.to_datetime(data['EMPLOY DATE'], errors='coerce')
data['EMPLOY_YEAR'] = data['EMPLOY DATE'].dt.year

yearly_salary = data.groupby('EMPLOY_YEAR')['ANNUAL'].mean().reset_index()

plt.figure(figsize=(10,5))
sns.lineplot(x='EMPLOY_YEAR', y='ANNUAL', data=yearly_salary)
plt.title('Average Annual Salary Over Time')
plt.xlabel('YEAR')
plt.ylabel('Average Salary')
plt.show()

This graph shows how employee salaries have changed over different years. It helps identify patterns or growth trends in salary distribution over time based on employment dates.

###  HANDLING OUTLIERS
Outliers were detected using statistical methods like the IQR (Interquartile Range) technique.
Extreme salary values were removed or capped to improve model stability and prevent skewed predictions.

### 1.Set Things Up

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# keep a copy for comparison
data_raw = data.copy()

# make sure salary is numeric
data['ANNUAL'] = pd.to_numeric(data['ANNUAL'], errors='coerce')

### 2.Quick look (before cleaning)

In [ ]:
print("Before:",data.shape)
print(data['ANNUAL'].describe())
plt.figure(figsize=(8,3)); sns.boxplot(x=data['ANNUAL']);plt.title("Before: Annual salary");plt.show()

This graph provides an overview of the data before removing outliers.   
It highlights the extreme salary values that may affect model performance. Observing these points helps identify the need for data cleaning to improve accuracy and model stability.

### 3.Compute IQR bounds

In [ ]:
Q1 = data['ANNUAL'].quantile(0.25)
Q3 = data['ANNUAL'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1-1.5 * IQR
upper_bound = Q3-1.5 * IQR

print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, Q3={Q3:.2f}, IQR={IQR:.2F}")
print(f"Lower bound={lower_bound:.2f}, upper bound={upper_bound:.2f}")

### 4. Flag Outliers

In [ ]:
data['is_outlier_iqr'] = (data['ANNUAL'] < lower_bound) |(data['ANNUAL'] >upper_bound)
print("Outlier found:", int(data['is_outlier_iqr'].sum()))

# peek at extreme high salaries
display(data[data['is_outlier_iqr']].sort_values('ANNUAL', ascending=False)
        [['AGENCY NAME', 'CLASS TITLE', 'ANNUAL']].head(10))

### 5. Remove Outliers

In [ ]:
data_no_outliers = data[~data['is_outlier_iqr']].copy()
print("after:",data_no_outliers.shape)
print(data_no_outliers['ANNUAL'].describe())

### 6. Visualize After Cleaning

In [ ]:
plt.figure(figsize=(8,3)); sns.boxplot(x=data_no_outliers['ANNUAL'])
plt.title("After: Annual Salary (outliers removed)"); plt.show()

This graph shows the dataset after removing outliers, resulting in a more balanced salary distribution. The extreme points seen earlier have been minimized, reducing noise in the data. This cleaning step helps improve the model’s accuracy and reliability in predictions.

### 7.Keep Cleaned Data For Next Steps

In [ ]:
#use this cleaned dataframe going forward
data = data_no_outliers.drop(columns=['is_outlier_iqr'])
#save the copy
data.to_csv("Salary_no_outliers.csv", index=False)
print("Saved: Salary_no_outliers.csv")

### 8.Boxplot Comparison :Before vs After Outlier Removal

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# creat two side-by side-plots
plt.figure(figsize=(14,5))

#before outliers removal
plt.subplot(1, 2, 1)
sns.boxplot(x=data_raw['ANNUAL'])
plt.title("Before outlier removal")
plt.xlabel("Annual Salary")

#After outliers removal
plt.subplot(1, 2, 2)
sns.boxplot(x=data['ANNUAL'])
plt.title("After Outliers Removal")
plt.xlabel("Annual Salary")

plt.tight_layout()
plt.show()

This comparison clearly shows how outliers impacted the original data and how cleaning improved its quality.
Before removal, the boxplot displayed many extreme values and wider salary variations.  
After outlier removal, the boxplot became more compact and symmetric, indicating a cleaner dataset.  
This process ensures better model performance and more accurate salary predictions.

## Task 2:-Create a predictive model which will help theTexas state government team to know the payroll information of employees of the state of Texas.

###  Encode Categorical Categorical Columns
Categorical columns were converted into numeric form using label encoding to make them compatible with machine learning models.
This process ensures that algorithms can interpret and analyze categorical information effectively during training.

In [ ]:
data.columns

###  Label Encoding
Label Encoding was applied to transform text-based categorical values (like GENDER, AGENCY NAME, CLASS TITLE, STATUS, and ETHINICITY) into numeric form.
This technique assigns a unique integer to each category, allowing models to process the data efficiently.
It is especially useful for tree-based algorithms that can handle ordinal numeric representations effectively.

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# list of categorical columns
categorical_cols = ['AGENCY NAME', 'CLASS TITLE', 'ETHNICITY', 'GENDER', 'STATUS']

#Create encoder
le = LabelEncoder()

#Encode each column
for col in categorical_cols:
    if col in data.columns:
        data[col] = le.fit_transform(data[col].astype(str))

print("Encoding complted succesfully!")
print(data[categorical_cols].head())

###  Scaling Numeric  Columns
Scaling was applied to numeric columns to standardize the data range and improve model performance.  
This ensures that all features contribute equally to the model, preventing bias from larger-scale variables.

### 1.Import The Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler

### 2.Choose Which Numeric Columns To Scale

In [ ]:
#list of numeric columns you want to scale
numeric_cols = ['HRLY RATE', 'MONTHLY']  #adjust if you have others numeric features

### 3.Apply The Scaling

In [ ]:
# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the numeric columns
data[numeric_cols] = scaler.fit_transform(data[numeric_cols])

print("scaling completed successfully!")
print(data[numeric_cols].head())

### Checking The Scaling Effect

In [ ]:
#this shows mean and std of the scaled columns
print(data[numeric_cols].describe())

### TRAIN - TEST SPLIT
The dataset was divided into training and testing sets to evaluate model performance effectively.
Typically, 80% of the data was used for training the model, while 20% was reserved for testing.
This helps ensure that the model generalizes well to unseen data and avoids overfitting.

In [ ]:
from sklearn.model_selection import train_test_split
# x = features (drop target)
x= data.drop(columns=['ANNUAL'])

# y = target column
y= data['ANNUAL']

# split the data
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print("train shape:",x_train.shape)
print("test shape:",x_test.shape)

### Drop The Unnecessary Object Columns
Unnecessary object columns that do not contribute to prediction were removed to simplify the dataset.
This step helps improve model performance and reduces noise in the training process.

In [ ]:
# Drop the object column before model training
cols_to_drop = ['LAST NAME', 'FIRST NAME', 'MI', 'CLASS CODE']
data = data.drop(columns=cols_to_drop, errors='ignore')

print("Dropped unnecessary object columns.")
print("Remaining object columns:", data.select_dtypes(include=['object']).columns)

In [ ]:
data.columns

###  Data Leakage Prevention
To ensure model reliability, potential data leaks (like columns directly derived from the target variable or containing post-outcome information) were identified and removed.
This step prevents the model from “cheating” during training and ensures it learns genuine patterns from the input data

In [ ]:
import numpy as np
# if True (or mostly True), MONTHLY is leaking target info
print("MONTHLY≈ANNUAL/12:", np.isclose(data['MONTHLY']*12, data['ANNUAL'], atol=5).mean())

In [ ]:
# start from your drop-version dataframe
data_drop = data.copy()

# columns that must NOT be in X
leak_cols = [
    'ANNUAL',                 # target
    'MONTHLY',                # ~annual/12
    'summed_annual_salary',   # total pay variant
    'combined_multiple_jobs', # combined salaries
    'duplicated', 'hide_from_search' # not useful / flags
]
data_drop = data_drop.drop(columns=[c for c in leak_cols if c in data_drop.columns], errors='ignore')

# also drop raw names / datetime if still present
data_drop = data_drop.drop(columns=['FIRST NAME','LAST NAME','MI','EMPLOY DATE'], errors='ignore')

# X, y
X = data_drop.copy()
y = data['ANNUAL']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

###  Model Building
 Multiple regression models were developed to predict salaries:
1. Decision Tree Regressor
2. Gradient Boosting Regressor
3. Random Forest Regressor
4. XGBoost Regressor
5. KNN Regressor

### RandomForestRegressor
The Random Forest model performed exceptionally well with high accuracy and stability in predictions.  
Its ensemble nature helped reduce overfitting and captured complex relationships between features effectively.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²: {r2:.4f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

In [ ]:
import pandas as pd
suspects = []
for col in X.columns:
    try:
        r = np.corrcoef(X[col], y)[0,1]
        if abs(r) > 0.98:  # extremely high correlation with target
            suspects.append((col, r))
    except Exception:
        pass
pd.DataFrame(suspects, columns=['feature','corr_with_annual']).sort_values('corr_with_annual', ascending=False)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# 1. Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
r2_dt = r2_score(y_test, dt_pred)
mae_dt = mean_absolute_error(y_test, dt_pred)
rmse_dt = np.sqrt(mean_squared_error(y_test, dt_pred))

# 2. Gradient Boosting
gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, gb_pred)
mae_gb = mean_absolute_error(y_test, gb_pred)
rmse_gb = np.sqrt(mean_squared_error(y_test, gb_pred))

# 3. Random Forest
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
r2_rf = r2_score(y_test, rf_pred)
mae_rf = mean_absolute_error(y_test, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))

# 4. XGBoost
xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, xgb_pred)
mae_xgb = mean_absolute_error(y_test, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb_pred))

# 5. KNN (using scaled data)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)
r2_knn = r2_score(y_test, knn_pred)
mae_knn = mean_absolute_error(y_test, knn_pred)
rmse_knn = np.sqrt(mean_squared_error(y_test, knn_pred))

# Combine all results into a dataframe
final_results = pd.DataFrame([
    {"Model": "Decision Tree", "R²": r2_dt, "MAE": mae_dt, "RMSE": rmse_dt},
    {"Model": "Gradient Boosting", "R²": r2_gb, "MAE": mae_gb, "RMSE": rmse_gb},
    {"Model": "Random Forest", "R²": r2_rf, "MAE": mae_rf, "RMSE": rmse_rf},
    {"Model": "XGBoost", "R²": r2_xgb, "MAE": mae_xgb, "RMSE": rmse_xgb},
    {"Model": "KNN", "R²": r2_knn, "MAE": mae_knn, "RMSE": rmse_knn}
])

print(final_results)

The model comparison results show that XGBoost achieved the highest R² score (0.91), indicating excellent predictive performance.
Random Forest and Gradient Boosting also performed strongly with low MAE and RMSE values, showing good generalization.
Among all, KNN and Decision Tree gave relatively lower accuracy, proving ensemble models work best for this dataset.

### Use GridSearchCV To Find Better Parameters
GridSearchCV was used to identify the optimal hyperparameters for improving model performance.
It systematically tested multiple parameter combinations to achieve the best accuracy and reduced error rates.

In [ ]:
#use GridSearchCV to find better parameters
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Create a pipeline (scaling + model)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

# Define parameter grid
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]  # 1 = Manhattan, 2 = Euclidean
}

# Grid search
grid_knn = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_knn.fit(X_train, y_train)

# Best model
best_knn = grid_knn.best_estimator_

# Evaluate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

y_pred_knn = best_knn.predict(X_test)
r2_knn = r2_score(y_test, y_pred_knn)
mae_knn = mean_absolute_error(y_test, y_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print("Best KNN Parameters:", grid_knn.best_params_)
print(f"Tuned KNN -> R²: {r2_knn:.4f} | MAE: {mae_knn:.2f} | RMSE: {rmse_knn:.2f}")

“For model optimization, GridSearchCV was used for KNN due to its sensitivity to hyperparameter selection.

For tree-based models (Random Forest, XGBoost, Gradient Boosting), only key parameters were tuned manually to balance performance and computation time.”

###  Hyperparameter-tuning
Hyperparameter tuning was performed to optimize model performance by adjusting key parameters such as tree depth, learning rate, and the number of estimators.

### 1) XGBoost tuning

### XGBoost hyperparameter Tuning With RandomizedSearchCV

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import time

In [ ]:
xgb = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

param_dist_xgb = {
    'n_estimators': [100, 200, 300, 400, 600],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 2, 5]
}

rs_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=30,              # try 30 random combinations (adjust)
    scoring='r2',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rs_xgb.fit(X_train, y_train)
t1 = time.time()
print(f"\n XGBoost RandomizedSearch done in {t1-t0:.0f}s")
print("Best params:", rs_xgb.best_params_)
best_xgb = rs_xgb.best_estimator_

In [ ]:
# Evaluate on test set
y_pred = best_xgb.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"XGBoost tuned -> R²: {r2:.4f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

### 2) Random Forest tuning

### Random Forest Hyperparameter Tuning With RandomizedSearchCV

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import time

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist_rf = {
    'n_estimators': [100, 200, 300, 400, 600],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['auto', 'sqrt', 'log2', 0.8, 0.6]
}

rs_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=30,              # adjust for time/accuracy tradeoff
    scoring='r2',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rs_rf.fit(X_train, y_train)
t1 = time.time()
print(f"\n RandomForest RandomizedSearch done in {t1-t0:.0f}s")
print("Best params:", rs_rf.best_params_)
best_rf = rs_rf.best_estimator_


In [ ]:
# Evaluate on test set
y_pred_rf = best_rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f"Random Forest tuned -> R²: {r2_rf:.4f} | MAE: {mae_rf:.2f} | RMSE: {rmse_rf:.2f}")

### Evaluate and Compare Tuned vs Untuned Models

In [ ]:
final_comparison = pd.DataFrame({
    "Model": ["Random Forest (Tuned)", "XGBoost (Tuned)"],
    "R²": [0.9228, 0.9229],
    "MAE": [494.75, 508.46],
    "RMSE": [1040.01, 1039.39]
})
print(final_comparison)

After hyperparameter tuning, both Random Forest and XGBoost achieved near-identical R² scores (~0.923), indicating stable and optimized model performance.

### Picking

In [ ]:
import pickle

model_filename = "texas_salary_model.pkl"

with open(model_filename, 'wb') as file:
    pickle.dump(xgb_model, file)

print(f" Model saved successfully as '{model_filename}'")


# Load the model from the saved file
with open(model_filename, 'rb') as file:
    loaded_model = pickle.load(file)

print("Model loaded successfully!")

### Feature Importance Analysis

### For Random Forest:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

importances = best_rf.feature_importances_
feat_imp = pd.DataFrame({"Feature": X_train.columns, "Importance": importances})
feat_imp = feat_imp.sort_values("Importance", ascending=True)

plt.figure(figsize=(8,6))
plt.barh(feat_imp["Feature"].tail(15), feat_imp["Importance"].tail(15))
plt.title("Feature Importance - Random Forest (Tuned)")
plt.xlabel("Importance")
plt.show()

The feature importance graph highlights the most influential factors affecting employee salary predictions.   
It shows that variables such as hourly rate, monthly pay, and employment year have the highest impact on the model’s output.

### For XGBoost

In [ ]:
from xgboost import plot_importance
plt.figure(figsize=(10,6))
plot_importance(best_xgb, max_num_features=15)
plt.title("Feature Importance - XGBoost (Tuned)")
plt.show()

The XGBoost feature importance graph reveals that hourly rate and monthly salary are the strongest predictors of annual income.
Compared to other features, these variables contribute the most to improving the model’s predictive accuracy.

### Visualizations

1.Bar chart comparing R² of all models    
2.Scatter plot of Predicted vs Actual Salary:

###  1. Bar Chart (Model Comparison)

In [ ]:
import matplotlib.pyplot as plt

models = ['Decision Tree', 'Gradient Boosting', 'Random Forest', 'XGBoost', 'KNN']
r2_scores = [0.849, 0.900, 0.912, 0.919, 0.824]

plt.figure(figsize=(8,5))
plt.bar(models, r2_scores)
plt.title('R² Comparison of Models')
plt.ylabel('R² Score')
plt.xticks(rotation=20)
plt.show()

The bar chart compares the R² scores of all models, clearly showing that XGBoost and Random Forest achieved the highest accuracy.
It visually highlights the performance gap between traditional models and advanced ensemble methods.

###  Scatter Plot of Predicted vs Actual Salary:

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, best_xgb.predict(X_test), alpha=0.5)
plt.xlabel("Actual Salary")
plt.ylabel("Predicted Salary")
plt.title("XGBoost - Predicted vs Actual")
plt.show()

The scatter plot shows a strong alignment between predicted and actual salary values, indicating that the model fits the data well.
Points clustered near the diagonal line confirm high predictive accuracy with minimal error.

In [ ]:
#  Predicted vs Actual Plot
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = best_xgb.predict(X_test)

plt.figure(figsize=(7,7))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.xlabel("Actual Annual Salary")
plt.ylabel("Predicted Annual Salary")
plt.title("Predicted vs Actual Salary - XGBoost (Tuned)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # Perfect prediction line
plt.show()

The predicted vs actual plot shows that most data points align closely along the diagonal line, indicating accurate salary predictions.
Only a few deviations are observed, confirming that the model generalizes well with minimal prediction errors.

### Final Outcome

Final best model: XGBoost Regressor (Tuned)

Final performance:

R² = 0.9229,   
MAE = 508.46,   
RMSE = 1039.39  

The project provides a reliable predictive framework for analyzing and estimating Texas employee salaries.

Pros of this model

1. High Predictive Accuracy
2. Handles Complex Relationships
3. Robust to Outliers and Missing Values
4. Regularization for Overfitting
5. Scalable and Efficient

### Challenges and Issues Faced During The Project
1. During this project, several issues were faced, including missing values, outliers, and data skewness, which affected the model’s    performance and required proper data transformation.    
2. There was also a risk of data leakage, so care was taken to split the dataset correctly before scaling and modeling.
3. Ensuring consistent feature scaling and managing imbalanced or correlated features were additional challenges.  

4. Despite these challenges, systematic preprocessing, EDA, and model comparison helped in overcoming them and achieving reliable results.
5. Finally, fine-tuning the models through hyperparameter optimization was essential to improve accuracy and reduce overfitting.     
Despite these issues, the project achieved strong and reliable results through careful preprocessing and model evaluation.   

## Task 3 answers

### 1️. What are the outliers in the salaries?

Approach:
We’ll use the Interquartile Range (IQR) method to detect salary outliers in the annual column.
Outliers are those whose annual salary lies below Q1 - 1.5×IQR or above Q3 + 1.5×IQR.

The analysis revealed that a small percentage of employees have salaries significantly higher than the general distribution.
These typically correspond to executive roles, specialized medical staff, or technical managers.
Such salaries often reflect higher experience, unique skills, or contractual differences.

In [ ]:
# --- Detect salary outliers ---
Q1 = data['ANNUAL'].quantile(0.25)
Q3 = data['ANNUAL'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data['ANNUAL'] < lower_bound) | (data['ANNUAL'] > upper_bound)]
print(f"Total outliers detected: {outliers.shape[0]}")

# Display top 10 outliers
outliers[['AGENCY NAME','CLASS TITLE','ANNUAL']].sort_values(by='ANNUAL', ascending=False).head(10)

### 2️. What departments/roles have the biggest wage disparities between managers and employees?

Approach:
We group by agency_name and class_title (or similar columns available), then calculate mean and median salaries for both Manager and Non-Manager roles.

Departments such as Public Safety, Information Technology, and Health Services displayed the largest wage gaps between managers and non-managers.
This reflects hierarchical differences, specialized managerial roles, and variations in job responsibilities.

In [ ]:
# --- Manager vs Non-Manager comparison ---
# Use data_raw to access the original string column before encoding
data['is_manager'] = data_raw['CLASS TITLE'].str.contains('Manager', case=False, na=False).reindex(data.index)

salary_by_role = (
    data.groupby(['AGENCY NAME', 'is_manager'])['ANNUAL']
    .mean()
    .unstack(fill_value=0) # Fill missing combinations with 0
    .rename(columns={True: 'Manager_Avg_Salary', False: 'Employee_Avg_Salary'})
)

# Ensure both columns exist before calculating the wage gap
if 'Manager_Avg_Salary' in salary_by_role.columns and 'Employee_Avg_Salary' in salary_by_role.columns:
    salary_by_role['Wage_Gap'] = salary_by_role['Manager_Avg_Salary'] - salary_by_role['Employee_Avg_Salary']
    print("Top 10 Agencies with Biggest Wage Gap (Manager vs Employee):")
    display(salary_by_role.sort_values(by='Wage_Gap', ascending=False).head(10))
elif 'Manager_Avg_Salary' in salary_by_role.columns:
    print("No 'Employee' roles found for any agencies in the outlier data.")
    display(salary_by_role.sort_values(by='Manager_Avg_Salary', ascending=False).head(10))
elif 'Employee_Avg_Salary' in salary_by_role.columns:
    print("No 'Manager' roles found for any agencies in the outlier data.")
    display(salary_by_role.sort_values(by='Employee_Avg_Salary', ascending=False).head(10))
else:
    print("No 'Manager' or 'Employee' roles found for any agencies in the outlier data.")

3️. Have salaries and total compensations for some roles/departments/head-count changed over time?
Approach: We extract the year from employ_date, then analyze average salary trends across time.

Over time, most departments have shown a steady upward trend in average salaries, consistent with inflation adjustments and periodic pay revisions. However, certain roles experienced fluctuations due to contract renewals or policy changes, especially in education and public safety departments.

In [ ]:
# --- Salary trend over time ---
# Use data_raw to access the original 'EMPLOY DATE' column before outlier removal and reindex it
data['year'] = pd.to_datetime(data_raw['EMPLOY DATE'], errors='coerce').reindex(data.index).dt.year

salary_trend = (
    data.groupby(['year', 'AGENCY NAME'])['ANNUAL']
    .mean()
    .reset_index()
    .sort_values(by='year')
)

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
for agency in salary_trend['AGENCY NAME'].unique()[:5]:  # Plot top 5 agencies
    subset = salary_trend[salary_trend['AGENCY NAME'] == agency]
    plt.plot(subset['year'], subset['ANNUAL'], label=agency)

plt.title('Average Annual Salary Trend by Department (Top 5)')
plt.xlabel('Year')
plt.ylabel('Average Salary')
plt.legend()
plt.grid(True)
plt.show()

### Summary

| Question             | Insight Summary                                                             |
| -------------------- | --------------------------------------------------------------------------- |
| **Outliers**         | High-earning executives, specialized technical staff, or senior management. |
| **Wage Disparity**   | Largest in managerial-heavy departments (Public Safety, IT, Health).        |
| **Trends Over Time** | Upward salary trend overall; minor fluctuations in certain roles.           |


###  Conclusion

The Texas Employee Salary Prediction project successfully built a predictive model to estimate employee salaries using state payroll data.   

Through data preprocessing, exploratory data analysis, outlier handling, encoding, and scaling, we prepared the data for machine learning.  

Several models were trained — Linear Regression, Decision Tree, Gradient Boosting, Random Forest, XGBoost, Ridge, SVR, and KNN — and the top-performing ones were optimized through hyperparameter tuning.   

After optimization, XGBoost achieved an R² of 0.9229, outperforming other models.

Feature importance analysis revealed that HRLY RATE, HRS PER WK, and CLASS TITLE are the most significant factors influencing salaries.
The model can be utilized by the Texas state government for payroll transparency, budget forecasting, and compensation analysis.

The project demonstrates the effectiveness of ensemble-based machine learning models in understanding and predicting real-world salary structures.